# Module 4: AgentCore Observability for E-Commerce Multi-Agent System

## Overview

In Module 3, we deployed our e-commerce multi-agent system to AgentCore Runtime **without observability**. In this module, we will deploy **NEW** observability-enabled agent runtimes that capture traces, spans, and GenAI events in CloudWatch.

**Key Difference**: Module 3 agents remain unchanged. This module creates **separate agent runtimes** with full observability.

In this module, we will:

1. **Understand BedrockAgentCoreApp** - The SDK that provides automatic OTEL instrumentation
2. **Build Observability-Enabled Agent Images** - Using `BedrockAgentCoreApp` pattern
3. **Create NEW Agent Runtimes** - Separate from Module 3, with full tracing enabled
4. **Test Multi-Agent Orchestration** - Verify functionality with observability
5. **Validate Traces in CloudWatch** - Verify traces, spans, and GenAI events are captured

## Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          AgentCore Runtimes                                  │
│                                                                              │
│  ┌─────────────────────────┐     ┌─────────────────────────────────────────┐│
│  │  Module 3 Agents        │     │  Module 4 Agents (Observability)        ││
│  │  (No Traces)            │     │  (Full OTEL Tracing)                    ││
│  │                         │     │                                          ││
│  │  • order_agent          │     │  • order_agent_observability      ◄──── ││ OTEL Traces
│  │  • product_agent        │     │  • product_agent_observability          ││
│  │  • account_agent        │     │  • account_agent_observability          ││
│  │  • orchestrator         │     │  • orchestrator_observability    ◄──── ││ OTEL Traces
│  └─────────────────────────┘     └─────────────────────────────────────────┘│
│                                              │                               │
│                                              │ MCP                           │
│                                              ▼                               │
│                          ┌──────────────────────────────┐                   │
│                          │    AgentCore Gateway         │                   │
│                          │  ┌───────┬───────┬───────┐   │                   │
│                          │  │Order  │Product│Account│   │                   │
│                          │  │Tools  │Tools  │Tools  │   │                   │
│                          │  └───────┴───────┴───────┘   │                   │
│                          └──────────────────────────────┘                   │
└─────────────────────────────────────────────────────────────────────────────┘
```

## What You'll See in CloudWatch GenAI Observability

- **invoke_agent** spans - Agent invocation with input/output
- **chat** spans - LLM model calls with token usage
- **execute_tool** spans - Tool calls with inputs/outputs
- **execute_event_loop_cycle** spans - Agent reasoning steps
- **gen_ai events** - Detailed GenAI semantic convention events

## Prerequisites

- ✅ Completed Module 3: Production Deployment (Gateway and tools configured)
- ✅ Docker running locally
- ✅ AWS credentials with AgentCore, ECR, and CloudWatch permissions

### Time: ~30 minutes


## Step 1: Setup and Configuration


In [1]:
# Install dependencies
!pip install -q boto3 bedrock-agentcore strands-agents aws-opentelemetry-distro

In [1]:
import boto3
import json
import time
import uuid
import os
import subprocess
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

# Load deployment info from Module 3
try:
    %store -r deployment_info
    %store -r REGION
    print(f"✅ Loaded deployment info from Module 3")
    print(f"   Region: {REGION}")
    print(f"   Gateway URL: {deployment_info.get('gateway_url', 'N/A')}")
except:
    print("⚠️ Could not load deployment info from Module 3")
    print("   Using default region")
    session = boto3.Session()
    REGION = session.region_name or "us-west-2"
    deployment_info = None

# Initialize AWS clients
logs_client = boto3.client("logs", region_name=REGION)
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)
xray_client = boto3.client("xray", region_name=REGION)
ecr_client = boto3.client("ecr", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"

print(f"\n📋 Configuration:")
print(f"   Account ID: {ACCOUNT_ID}")
print(f"   Region: {REGION}")
print(f"   ECR Registry: {ECR_REGISTRY}")

✅ Loaded deployment info from Module 3
   Region: us-west-2
   Gateway URL: https://ecommerce-workshop-product-gateway-uh7wpjkjex.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp

📋 Configuration:
   Account ID: 881490113957
   Region: us-west-2
   ECR Registry: 881490113957.dkr.ecr.us-west-2.amazonaws.com


In [2]:
# Get runtime role ARN from Module 3 (we'll reuse the same role)
def get_runtime_role_arn():
    """Get the runtime role ARN created in Module 3."""
    role_name = "ecommerce-workshop-runtime-role"
    try:
        response = iam_client.get_role(RoleName=role_name)
        return response["Role"]["Arn"]
    except Exception as e:
        print(f"Error getting role: {e}")
        return None


RUNTIME_ROLE_ARN = get_runtime_role_arn()

# Define new agent runtime names with observability suffix
# Note: orchestrator omits "_agent_" due to 48 char name limit
OBSERVABILITY_AGENT_NAMES = {
    "order": "ecommerce_workshop_order_agent_observability",  # 44 chars
    "product": "ecommerce_workshop_product_agent_observability",  # 46 chars
    "account": "ecommerce_workshop_account_agent_observability",  # 46 chars
    "orchestrator": "ecommerce_workshop_orchestrator_observability",  # 44 chars (no _agent_ due to length)
}

print("=" * 60)
print("MODULE 4: CREATING NEW OBSERVABILITY-ENABLED AGENTS")
print("=" * 60)
print(f"\n✅ Runtime Role: {RUNTIME_ROLE_ARN}")
print("\n📋 New Agent Runtimes to Create:")
for agent_type, name in OBSERVABILITY_AGENT_NAMES.items():
    print(f"   • {name} ({len(name)} chars)")

print("\n💡 Note: Module 3 agents will remain unchanged.")

MODULE 4: CREATING NEW OBSERVABILITY-ENABLED AGENTS

✅ Runtime Role: arn:aws:iam::881490113957:role/ecommerce-workshop-runtime-role

📋 New Agent Runtimes to Create:
   • ecommerce_workshop_order_agent_observability (44 chars)
   • ecommerce_workshop_product_agent_observability (46 chars)
   • ecommerce_workshop_account_agent_observability (46 chars)
   • ecommerce_workshop_orchestrator_observability (45 chars)

💡 Note: Module 3 agents will remain unchanged.


## Step 2: Verify CloudWatch Transaction Search

CloudWatch Transaction Search must be enabled to view AgentCore traces and spans. This is a prerequisite for GenAI Observability.


In [3]:
def check_transaction_search_enabled():
    """
    Check if CloudWatch Transaction Search is enabled by querying X-Ray traces.
    """
    print("Checking CloudWatch Transaction Search status...\n")

    try:
        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(hours=1)

        response = xray_client.get_trace_summaries(
            StartTime=start_time, EndTime=end_time, Sampling=True
        )

        print("✅ CloudWatch Transaction Search is ENABLED")
        return True

    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if "ResourceNotFoundException" in error_code:
            print("❌ CloudWatch Transaction Search is NOT ENABLED")
            print(
                f"\n1. Go to: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:settings/transaction-search"
            )
            print("2. Click 'Edit'")
            print("3. Enable 'Ingest spans as structured logs in OpenTelemetry format'")
            print("4. Click 'Save'")
            return False
        elif "AccessDenied" in str(e):
            print("⚠️ Cannot verify - insufficient IAM permissions for X-Ray")
            return True
        else:
            print(f"⚠️ Unexpected error: {error_code}")
            return True
    except Exception as e:
        print(f"⚠️ Error checking status: {e}")
        return True


TRANSACTION_SEARCH_OK = check_transaction_search_enabled()

Checking CloudWatch Transaction Search status...

✅ CloudWatch Transaction Search is ENABLED


## Step 3: Configure X-Ray Permissions for Runtime Role

**CRITICAL**: The agent runtime role needs X-Ray permissions to export OTEL traces. Without these permissions, the OTLP exporter will fail with "403 - not authorized to perform xray:PutTraceSegments".


In [4]:
# Configure X-Ray permissions for the runtime role
RUNTIME_ROLE_NAME = "ecommerce-workshop-runtime-role"


def configure_xray_permissions(role_name: str) -> bool:
    print(f"Configuring X-Ray permissions for role: {role_name}\n")
    xray_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "xray:PutTraceSegments",
                    "xray:PutTelemetryRecords",
                    "xray:GetSamplingRules",
                    "xray:GetSamplingTargets",
                    "xray:GetSamplingStatisticSummaries",
                ],
                "Resource": "*",
            }
        ],
    }
    try:
        try:
            iam_client.get_role_policy(
                RoleName=role_name, PolicyName="xray-tracing-policy"
            )
            print("ℹ️  X-Ray policy already exists on role")
            return True
        except ClientError as e:
            if e.response["Error"]["Code"] != "NoSuchEntity":
                raise
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName="xray-tracing-policy",
            PolicyDocument=json.dumps(xray_policy),
        )
        print("✅ X-Ray permissions added to runtime role")
        return True
    except ClientError as e:
        print(f"❌ Error: {e}")
        return False


XRAY_PERMISSIONS_OK = configure_xray_permissions(RUNTIME_ROLE_NAME)

Configuring X-Ray permissions for role: ecommerce-workshop-runtime-role

ℹ️  X-Ray policy already exists on role


## Step 4: Build Observability-Enabled Agent Images

### Understanding the BedrockAgentCoreApp Pattern

The agent scripts use `BedrockAgentCoreApp` instead of FastAPI:

- `@app.entrypoint` decorator creates OTEL spans for each invocation
- `app.run()` provides `/invocations` and `/ping` endpoints
- `opentelemetry-instrument` wrapper enables auto-instrumentation
- Strands Agent creates spans for tool calls and LLM invocations

Now we build Docker images for all 4 agents using the `BedrockAgentCoreApp` pattern with OTEL instrumentation enabled.


In [5]:
# Workshop prefix and ECR repository configuration
WORKSHOP_PREFIX = "ecommerce-workshop"

ECR_REPOS = {
    "order": f"{WORKSHOP_PREFIX}-order-agent",
    "product": f"{WORKSHOP_PREFIX}-product-agent",
    "account": f"{WORKSHOP_PREFIX}-account-agent",
    "orchestrator": f"{WORKSHOP_PREFIX}-orchestrator-agent",
}

AGENT_FILES = {
    "order": "order_agent.py",
    "product": "product_agent.py",
    "account": "account_agent.py",
    "orchestrator": "orchestrator_agent.py",
}

AGENTS_DIR = os.path.join(os.getcwd(), "agents")

print("Agent Configuration:")
print("=" * 60)
for agent_type, repo in ECR_REPOS.items():
    print(f"\n{agent_type.upper()}:")
    print(f"  File: {AGENT_FILES[agent_type]}")
    print(f"  ECR: {ECR_REGISTRY}/{repo}:observability")

# Authenticate with ECR
print("\n" + "=" * 60)
print("Authenticating with ECR...")
!aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}
print("✅ ECR authentication successful")

Agent Configuration:

ORDER:
  File: order_agent.py
  ECR: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

PRODUCT:
  File: product_agent.py
  ECR: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

ACCOUNT:
  File: account_agent.py
  ECR: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-account-agent:observability

ORCHESTRATOR:
  File: orchestrator_agent.py
  ECR: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-orchestrator-agent:observability

Authenticating with ECR...
Login Succeeded!
✅ ECR authentication successful


In [6]:
# Create ECR repositories if they don't exist
print("=" * 60)
print("CREATING ECR REPOSITORIES")
print("=" * 60)

for agent_type, repo_name in ECR_REPOS.items():
    try:
        # Check if repository exists
        ecr_client.describe_repositories(repositoryNames=[repo_name])
        print(f"✅ {agent_type.upper()}: Repository '{repo_name}' already exists")
    except ecr_client.exceptions.RepositoryNotFoundException:
        # Create repository
        try:
            response = ecr_client.create_repository(repositoryName=repo_name)
            repo_uri = response["repository"]["repositoryUri"]
            print(f"✅ {agent_type.upper()}: Created repository '{repo_name}'")
            print(f"   URI: {repo_uri}")
        except Exception as e:
            print(
                f"❌ {agent_type.upper()}: Failed to create repository '{repo_name}': {e}"
            )
    except Exception as e:
        print(f"⚠️ {agent_type.upper()}: Error checking repository: {e}")

print("\n✅ ECR repository setup complete")

CREATING ECR REPOSITORIES
✅ ORDER: Repository 'ecommerce-workshop-order-agent' already exists
✅ PRODUCT: Repository 'ecommerce-workshop-product-agent' already exists
✅ ACCOUNT: Repository 'ecommerce-workshop-account-agent' already exists
✅ ORCHESTRATOR: Repository 'ecommerce-workshop-orchestrator-agent' already exists

✅ ECR repository setup complete


### Create ECR Repositories

Before building and pushing Docker images, we need to ensure that ECR repositories exist for all four agent types. Module 3 created only the product catalog agent repository, so we'll create the additional repositories needed for the multi-agent system.


In [7]:
# Build observability-enabled Docker images for all agents
import subprocess


def build_and_push_agent_image(agent_type: str, agent_file: str, ecr_repo: str) -> str:
    """Build and push a Docker image for an agent with observability enabled."""
    image_uri = f"{ECR_REGISTRY}/{ecr_repo}:observability"

    print(f"\n{'=' * 60}")
    print(f"BUILDING: {agent_type.upper()} AGENT")
    print(f"{'=' * 60}")
    print(f"  Agent file: {agent_file}")
    print(f"  Image URI: {image_uri}")

    try:
        # Build the image
        build_cmd = [
            "docker",
            "build",
            "--platform",
            "linux/arm64",
            "--build-arg",
            f"AGENT_FILE={agent_file}",
            "-t",
            image_uri,
            AGENTS_DIR,
        ]

        print(f"\n  Building image...")
        result = subprocess.run(build_cmd, capture_output=True, text=True, timeout=300)

        if result.returncode != 0:
            print(f"  ❌ Build failed: {result.stderr[:200]}")
            return None

        print(f"  ✅ Build successful")

        # Push to ECR
        print(f"  Pushing to ECR...")
        push_cmd = ["docker", "push", image_uri]
        result = subprocess.run(push_cmd, capture_output=True, text=True, timeout=300)

        if result.returncode != 0:
            print(f"  ❌ Push failed: {result.stderr[:200]}")
            return None

        print(f"  ✅ Push successful")
        return image_uri

    except subprocess.TimeoutExpired:
        print(f"  ❌ Timeout during build/push")
        return None
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return None


# Build all agent images
BUILD_RESULTS = {}

print("=" * 60)
print("BUILDING OBSERVABILITY-ENABLED AGENT IMAGES")
print("=" * 60)
print(f"Source directory: {AGENTS_DIR}")

for agent_type in ["order", "product", "account", "orchestrator"]:
    image_uri = build_and_push_agent_image(
        agent_type=agent_type,
        agent_file=AGENT_FILES[agent_type],
        ecr_repo=ECR_REPOS[agent_type],
    )
    BUILD_RESULTS[agent_type] = image_uri

# Summary
print("\n" + "=" * 60)
print("BUILD SUMMARY")
print("=" * 60)
successful = sum(1 for v in BUILD_RESULTS.values() if v)
print(f"\n{successful}/{len(BUILD_RESULTS)} images built successfully\n")
for agent_type, uri in BUILD_RESULTS.items():
    status = "✅" if uri else "❌"
    print(f"  {status} {agent_type}: {uri or 'FAILED'}")

BUILDING OBSERVABILITY-ENABLED AGENT IMAGES
Source directory: /Users/nfmoore/Documents/Development/Repository/ecommerce-agent-workshop/04-online-eval-observability/agents

BUILDING: ORDER AGENT
  Agent file: order_agent.py
  Image URI: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: PRODUCT AGENT
  Agent file: product_agent.py
  Image URI: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: ACCOUNT AGENT
  Agent file: account_agent.py
  Image URI: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-account-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: ORCHESTRATOR AGENT
  Agent file: orchestrator_agent.py
  Image URI: 881490113957.dkr.ecr.us-wes

## Step 5: Retrieve Gateway Configuration

**CRITICAL**: Agents need Gateway connection details (URL, User Pool ID, Client ID) to access MCP tools via the AgentCore Gateway.


In [10]:
# Retrieve Gateway configuration from Module 3 deployment
def get_gateway_configuration() -> dict:
    """
    Get Gateway configuration required for agent MCP tool access.
    Tries deployment_info first, then queries the Gateway API.
    """
    config = {"gateway_url": None, "user_pool_id": None, "client_id": None}

    print("Retrieving Gateway configuration...\n")

    # Step 1: Get what we can from deployment_info
    if deployment_info:
        print("Checking deployment_info from Module 3...")
        config["gateway_url"] = deployment_info.get("gateway_url")
        config["user_pool_id"] = deployment_info.get("user_pool_id")
        # Module 3 stores this as 'user_client_id', not 'client_id'
        config["client_id"] = deployment_info.get("client_id") or deployment_info.get(
            "user_client_id"
        )

        print(f"  gateway_url: {'✅' if config['gateway_url'] else '❌'}")
        print(f"  user_pool_id: {'✅' if config['user_pool_id'] else '❌'}")
        print(f"  client_id: {'✅' if config['client_id'] else '❌'}")

        if all(config.values()):
            print("\n✅ Gateway configuration loaded from Module 3 deployment info")
            return config

    # Step 2: Query Gateway API for missing values
    if not all(config.values()):
        print("\nQuerying Gateway API for missing configuration...")
        try:
            gateways = agentcore_control_client.list_gateways()

            for gw in gateways.get("gateways", []):
                if "ecommerce" in gw.get("gatewayName", "").lower():
                    gateway_id = gw["gatewayId"]
                    print(f"  Found gateway: {gateway_id}")

                    # Get detailed Gateway info
                    gw_details = agentcore_control_client.get_gateway(
                        gatewayIdentifier=gateway_id
                    )

                    # Extract Gateway URL if missing
                    if not config["gateway_url"]:
                        gateway_url = gw_details.get("gatewayUrl", "")
                        if not gateway_url:
                            gateway_name = gw_details.get("gatewayName", "")
                            gateway_url = f"https://{gateway_name}.gateway.bedrock-agentcore.{REGION}.amazonaws.com/mcp"
                        config["gateway_url"] = gateway_url

                    # Extract auth configuration
                    auth_config = gw_details.get("authorizerConfiguration", {})
                    print(f"  Auth config keys: {list(auth_config.keys())}")

                    if "customJWTAuthorizer" in auth_config:
                        jwt_auth = auth_config["customJWTAuthorizer"]

                        # Get user_pool_id if missing
                        if not config["user_pool_id"]:
                            config["user_pool_id"] = jwt_auth.get("userPoolId")

                        # Get client_id from allowed clients
                        allowed_clients = jwt_auth.get("allowedClients", [])
                        print(f"  Allowed clients: {allowed_clients}")
                        if allowed_clients and not config["client_id"]:
                            config["client_id"] = allowed_clients[0]
                            print(f"  Using client_id: {config['client_id']}")

                    break

        except Exception as e:
            print(f"⚠️ Error querying Gateway API: {e}")
            import traceback

            traceback.print_exc()

    return config


GATEWAY_CONFIG = get_gateway_configuration()

print("\n" + "=" * 60)
print("GATEWAY CONFIGURATION")
print("=" * 60)
if all(GATEWAY_CONFIG.values()):
    print(f"\n✅ Gateway URL: {GATEWAY_CONFIG['gateway_url'][:60]}...")
    print(f"✅ User Pool ID: {GATEWAY_CONFIG['user_pool_id']}")
    print(f"✅ Client ID: {GATEWAY_CONFIG['client_id']}")
else:
    print("\n❌ ERROR: Gateway configuration incomplete!")
    print("   Please ensure Module 3 was completed successfully.")
    for key, value in GATEWAY_CONFIG.items():
        status = "✅" if value else "❌"
        print(f"   {status} {key}: {value or 'MISSING'}")

Retrieving Gateway configuration...

Checking deployment_info from Module 3...
  gateway_url: ✅
  user_pool_id: ✅
  client_id: ✅

✅ Gateway configuration loaded from Module 3 deployment info

GATEWAY CONFIGURATION

✅ Gateway URL: https://ecommerce-workshop-product-gateway-uh7wpjkjex.gatewa...
✅ User Pool ID: us-west-2_g5Px4wTJy
✅ Client ID: 1nfuehtomuhp0jsskb4lroatoi


## Step 6: Deploy Observability-Enabled Agent Runtimes

This step deploys observability-enabled agents using a **non-destructive approach**:

- **Existing agents**: Uses `UpdateAgentRuntime` API to deploy a new version with updated image and configuration
- **New agents**: Uses `CreateAgentRuntime` API to create fresh agent runtimes

**Benefits of this approach:**

- Preserves agent runtime IDs and ARNs
- Maintains CloudWatch log delivery configurations
- Creates new versions for easy rollback
- No downtime from delete/recreate cycles

**Configuration includes:**

- **Gateway environment variables** (GATEWAY_URL, USER_POOL_ID, CLIENT_ID) for MCP tool access
- **OTEL environment variables** for trace emission and CloudWatch integration
- **GenAI semantic conventions** (`gen_ai_tool_definitions`) for detailed LLM and tool call events


In [11]:
def get_agent_env_vars(agent_type: str, gateway_config: dict) -> dict:
    """Get environment variables for an agent runtime."""
    # Derive OTEL service name from runtime name (underscores -> dashes)
    # This ensures consistency between runtime name and OTEL service name
    runtime_name = OBSERVABILITY_AGENT_NAMES[agent_type]
    service_name = runtime_name.replace("_", "-")

    return {
        # Gateway configuration for MCP tools
        "GATEWAY_URL": gateway_config.get("gateway_url", ""),
        "USER_POOL_ID": gateway_config.get("user_pool_id", ""),
        "CLIENT_ID": gateway_config.get("client_id", ""),
        "AGENT_REGION": REGION,
        "MODEL_ID": "global.anthropic.claude-sonnet-4-20250514-v1:0"
        if agent_type == "orchestrator"
        else "global.anthropic.claude-haiku-4-5-20251001-v1:0",
        # OTEL configuration for observability
        "AGENT_OBSERVABILITY_ENABLED": "true",
        "OTEL_PYTHON_DISTRO": "aws_distro",
        "OTEL_PYTHON_CONFIGURATOR": "aws_configurator",
        "OTEL_TRACES_EXPORTER": "otlp",
        "OTEL_LOGS_EXPORTER": "otlp",
        "OTEL_METRICS_EXPORTER": "none",
        "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
        "OTEL_SERVICE_NAME": service_name,
        # CRITICAL: Traces endpoint for CloudWatch GenAI Observability
        "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT": f"https://xray.{REGION}.amazonaws.com/v1/traces",
        "OTEL_EXPORTER_OTLP_LOGS_ENDPOINT": f"https://logs.{REGION}.amazonaws.com/v1/logs",
        # NOTE: Do NOT use gen_ai_latest_experimental - CloudWatch expects standard event format
        "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_tool_definitions",
    }


def get_existing_agent_by_name(agent_name: str) -> dict | None:
    """
    Find an existing agent runtime by name.
    Handles pagination to search through all agents.
    Returns runtime info dict if found, None otherwise.
    """
    try:
        next_token = None
        page_count = 0

        while True:
            page_count += 1

            # Build request parameters
            params = {}
            if next_token:
                params["nextToken"] = next_token

            response = agentcore_control_client.list_agent_runtimes(**params)
            runtimes = response.get("agentRuntimes", [])

            print(f"    Page {page_count}: checking {len(runtimes)} agents...")

            for runtime in runtimes:
                runtime_name = runtime.get("agentRuntimeName", "")
                if runtime_name == agent_name:
                    print(f"    Found matching agent: {runtime_name}")
                    return {
                        "name": runtime_name,
                        "id": runtime["agentRuntimeId"],
                        "arn": runtime["agentRuntimeArn"],
                        "status": runtime.get("status", "UNKNOWN"),
                    }

            # Check for more pages
            next_token = response.get("nextToken")
            if not next_token:
                break

        print(f"    Agent '{agent_name}' not found after {page_count} page(s)")

    except Exception as e:
        print(f"  ⚠️ Error checking for existing agent: {e}")
    return None


def deploy_observability_agent(
    agent_type: str, agent_name: str, image_uri: str, gateway_config: dict
) -> dict:
    """
    Deploy an observability agent using try-create-then-update pattern.

    1. Try to CREATE the agent
    2. If ConflictException (already exists), find the agent and UPDATE it

    This is more efficient than listing all agents upfront.
    """
    print(f"\n{'=' * 60}")
    print(f"DEPLOYING: {agent_type.upper()} AGENT (Observability)")
    print(f"{'=' * 60}")
    print(f"Agent Name: {agent_name}")
    print(f"Image URI: {image_uri}")

    env_vars = get_agent_env_vars(agent_type, gateway_config)

    # Step 1: Try to CREATE the agent
    try:
        print(f"\n  Attempting to create new agent...")

        response = agentcore_control_client.create_agent_runtime(
            agentRuntimeName=agent_name,
            roleArn=RUNTIME_ROLE_ARN,
            agentRuntimeArtifact={
                "containerConfiguration": {"containerUri": image_uri}
            },
            networkConfiguration={"networkMode": "PUBLIC"},
            environmentVariables=env_vars,
        )

        runtime_id = response.get("agentRuntimeId")
        runtime_arn = response.get("agentRuntimeArn")

        print(f"\n✅ {agent_type.upper()} Agent CREATED successfully!")
        print(f"   Runtime ID: {runtime_id}")

        return {
            "name": agent_name,
            "id": runtime_id,
            "arn": runtime_arn,
            "status": "CREATING",
            "version": "1",
            "action": "CREATED",
        }

    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        error_msg = e.response["Error"]["Message"]

        # Step 2: If agent already exists, UPDATE it instead
        if "ConflictException" in error_code or "already exists" in error_msg.lower():
            print(f"  Agent already exists - switching to UPDATE...")
            print(f"  Searching for existing agent...")

            # Find the existing agent
            existing = get_existing_agent_by_name(agent_name)
            if not existing:
                print(f"\n❌ Error: Agent exists but couldn't retrieve details")
                return None

            runtime_id = existing["id"]
            print(f"  Found existing agent: {runtime_id}")

            # Update the existing agent
            try:
                print(f"  Updating with new image and configuration...")

                update_response = agentcore_control_client.update_agent_runtime(
                    agentRuntimeId=runtime_id,
                    roleArn=RUNTIME_ROLE_ARN,
                    agentRuntimeArtifact={
                        "containerConfiguration": {"containerUri": image_uri}
                    },
                    networkConfiguration={"networkMode": "PUBLIC"},
                    environmentVariables=env_vars,
                )

                new_version = update_response.get("agentRuntimeVersion", "N/A")

                print(f"\n✅ {agent_type.upper()} Agent UPDATED successfully!")
                print(f"   Runtime ID: {runtime_id}")
                print(f"   New Version: {new_version}")

                return {
                    "name": agent_name,
                    "id": runtime_id,
                    "arn": update_response.get("agentRuntimeArn", existing["arn"]),
                    "status": update_response.get("status", "UPDATING"),
                    "version": new_version,
                    "action": "UPDATED",
                }

            except Exception as update_error:
                print(f"\n❌ Error updating agent: {update_error}")
                return None
        else:
            print(f"\n❌ Error creating agent: {error_code}: {error_msg}")
            return None

    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        return None

In [12]:
# Deploy all observability-enabled agent runtimes using try-create-then-update pattern
OBSERVABILITY_RUNTIMES = {}

if not all(GATEWAY_CONFIG.values()):
    print("❌ ERROR: Gateway configuration incomplete!")
    print("   Please ensure Module 3 was completed successfully.")
else:
    print("=" * 60)
    print("DEPLOYING OBSERVABILITY AGENTS")
    print("=" * 60)
    print("\nStrategy: Try CREATE, if exists then UPDATE")
    print(f"\nGateway configuration validated ✅")
    print(f"  Gateway URL: {GATEWAY_CONFIG['gateway_url'][:50]}...")
    print(f"  User Pool ID: {GATEWAY_CONFIG['user_pool_id']}")
    print(f"  Client ID: {GATEWAY_CONFIG['client_id']}")

    for agent_type in ["order", "product", "account", "orchestrator"]:
        if not BUILD_RESULTS.get(agent_type):
            print(f"\n⚠️ Skipping {agent_type} - build failed")
            continue

        runtime_info = deploy_observability_agent(
            agent_type=agent_type,
            agent_name=OBSERVABILITY_AGENT_NAMES[agent_type],
            image_uri=BUILD_RESULTS[agent_type],
            gateway_config=GATEWAY_CONFIG,
        )

        if runtime_info:
            OBSERVABILITY_RUNTIMES[agent_type] = runtime_info

        time.sleep(2)

# Summary
print("\n" + "=" * 60)
print("DEPLOYMENT SUMMARY")
print("=" * 60)
successful = len(OBSERVABILITY_RUNTIMES)
print(f"\n{successful}/4 observability agents deployed\n")

created_count = sum(
    1 for info in OBSERVABILITY_RUNTIMES.values() if info.get("action") == "CREATED"
)
updated_count = sum(
    1 for info in OBSERVABILITY_RUNTIMES.values() if info.get("action") == "UPDATED"
)

print(f"  🆕 Created: {created_count}")
print(f"  📦 Updated: {updated_count}\n")

for agent_type, info in OBSERVABILITY_RUNTIMES.items():
    action = info.get("action", "UNKNOWN")
    version = info.get("version", "N/A")
    symbol = "🆕" if action == "CREATED" else "📦"
    print(f"  {symbol} {agent_type.upper()}: {info['id']} (v{version})")

DEPLOYING OBSERVABILITY AGENTS

Strategy: Try CREATE, if exists then UPDATE

Gateway configuration validated ✅
  Gateway URL: https://ecommerce-workshop-product-gateway-uh7wpjk...
  User Pool ID: us-west-2_g5Px4wTJy
  Client ID: 1nfuehtomuhp0jsskb4lroatoi

DEPLOYING: ORDER AGENT (Observability)
Agent Name: ecommerce_workshop_order_agent_observability
Image URI: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

  Attempting to create new agent...

✅ ORDER Agent CREATED successfully!
   Runtime ID: ecommerce_workshop_order_agent_observability-6Av3mF4GAl

DEPLOYING: PRODUCT AGENT (Observability)
Agent Name: ecommerce_workshop_product_agent_observability
Image URI: 881490113957.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

  Attempting to create new agent...

✅ PRODUCT Agent CREATED successfully!
   Runtime ID: ecommerce_workshop_product_agent_observability-ogrVv591RS

DEPLOYING: ACCOUNT AGENT (Observability)
Agent 

In [13]:
# Wait for agents to become ready after deployment
print("Waiting for observability agents to become READY...")
print("(New agents ~60-90s, updates ~30-60s)\n")

agents_ready = False
max_wait = 180  # 3 minutes max
start_time = time.time()

while time.time() - start_time < max_wait:
    all_ready = True
    status_updates = []

    for agent_type, info in OBSERVABILITY_RUNTIMES.items():
        try:
            response = agentcore_control_client.get_agent_runtime(
                agentRuntimeId=info["id"]
            )
            status = response.get("status", "UNKNOWN")
            OBSERVABILITY_RUNTIMES[agent_type]["status"] = status
            OBSERVABILITY_RUNTIMES[agent_type]["arn"] = response.get(
                "agentRuntimeArn", info.get("arn")
            )

            if status not in ["READY"]:
                all_ready = False
                status_updates.append(f"  {agent_type}: {status}")
        except Exception as e:
            all_ready = False
            status_updates.append(f"  {agent_type}: ERROR - {e}")

    if all_ready:
        agents_ready = True
        break

    if status_updates:
        elapsed = int(time.time() - start_time)
        print(f"[{elapsed}s] Still waiting...")
        for update in status_updates:
            print(update)

    time.sleep(15)

if agents_ready:
    print("\n✅ All observability agents are READY!")
    for agent_type, info in OBSERVABILITY_RUNTIMES.items():
        version = info.get("version", "N/A")
        print(f"   {agent_type}: {info['id']} (v{version}) - {info['status']}")
else:
    print("\n⚠️ Some agents may still be initializing. Proceeding anyway...")
    for agent_type, info in OBSERVABILITY_RUNTIMES.items():
        print(f"   {agent_type}: {info['id']} - {info.get('status', 'UNKNOWN')}")

Waiting for observability agents to become READY...
(New agents ~60-90s, updates ~30-60s)


✅ All observability agents are READY!
   order: ecommerce_workshop_order_agent_observability-6Av3mF4GAl (v1) - READY
   product: ecommerce_workshop_product_agent_observability-ogrVv591RS (v1) - READY
   account: ecommerce_workshop_account_agent_observability-UqmBBEE524 (v1) - READY
   orchestrator: ecommerce_workshop_orchestrator_observability-QMnQP48nWG (v1) - READY


## Step 7: Configure CloudWatch Log Delivery for Traces

**CRITICAL STEP**: To enable observability, we must configure CloudWatch Log Delivery to route OTEL traces from AgentCore to X-Ray. Without this configuration, agents emit telemetry but it doesn't appear in CloudWatch.

This creates:

1. **Delivery Source** - Links each agent runtime as a source of TRACES
2. **Delivery Destination** - Routes traces to X-Ray
3. **Delivery Connection** - Connects the source to the destination


In [14]:
def configure_log_delivery_for_agent(agent_type: str, runtime_info: dict) -> dict:
    """
    Configure CloudWatch Log Delivery for BOTH:
    1. APPLICATION_LOGS → CloudWatch Log Group (required for GenAI events)
    2. TRACES → X-Ray (required for spans)
    """
    runtime_id = runtime_info["id"]
    runtime_arn = runtime_info["arn"]

    # Extract short suffix for delivery names (max 60 chars)
    short_suffix = runtime_id.split("-")[-1] if "-" in runtime_id else runtime_id[-10:]

    print(f"\n{'=' * 60}")
    print(f"CONFIGURING LOG DELIVERY: {agent_type.upper()} AGENT")
    print(f"{'=' * 60}")
    print(f"  Runtime ID: {runtime_id}")
    print(f"  Short suffix: {short_suffix}")

    results = {
        "log_group": None,
        "logs_source": None,
        "logs_destination": None,
        "logs_delivery": None,
        "traces_source": None,
        "traces_destination": None,
        "traces_delivery": None,
    }

    try:
        # Step 0: Create vended log group for APPLICATION_LOGS
        vended_log_group = f"/aws/vendedlogs/bedrock-agentcore/{runtime_id}"
        log_group_arn = (
            f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{vended_log_group}"
        )

        try:
            logs_client.create_log_group(logGroupName=vended_log_group)
            print(f"  ✅ Created vended log group: {vended_log_group}")
            results["log_group"] = vended_log_group
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
                print(f"  ℹ️  Vended log group already exists")
                results["log_group"] = vended_log_group
            else:
                raise

        # ============================================================
        # Step 1: Configure APPLICATION_LOGS delivery (for GenAI events)
        # ============================================================
        print(f"\n  --- APPLICATION_LOGS (for GenAI events) ---")

        # Create logs delivery source
        logs_source_name = f"{agent_type}-{short_suffix}-logs-src"
        print(f"  Source name: {logs_source_name} ({len(logs_source_name)} chars)")
        try:
            logs_client.put_delivery_source(
                name=logs_source_name,
                resourceArn=runtime_arn,
                logType="APPLICATION_LOGS",
            )
            print(f"  ✅ Created APPLICATION_LOGS delivery source")
            results["logs_source"] = logs_source_name
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Logs source already exists")
                results["logs_source"] = logs_source_name
            else:
                raise

        # Create logs delivery destination (CloudWatch Log Group)
        logs_dest_name = f"{agent_type}-{short_suffix}-logs-dst"
        try:
            logs_client.put_delivery_destination(
                name=logs_dest_name,
                deliveryDestinationType="CWL",
                deliveryDestinationConfiguration={
                    "destinationResourceArn": log_group_arn
                },
            )
            print(f"  ✅ Created APPLICATION_LOGS delivery destination (CWL)")
            results["logs_destination"] = logs_dest_name
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Logs destination already exists")
                results["logs_destination"] = logs_dest_name
            else:
                raise

        # Create logs delivery connection
        try:
            dest_info = logs_client.get_delivery_destination(name=logs_dest_name)
            dest_arn = dest_info["deliveryDestination"]["arn"]
            logs_client.create_delivery(
                deliverySourceName=logs_source_name, deliveryDestinationArn=dest_arn
            )
            print(f"  ✅ Created APPLICATION_LOGS delivery connection")
            results["logs_delivery"] = "created"
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Logs delivery connection already exists")
                results["logs_delivery"] = "existing"
            else:
                raise

        # ============================================================
        # Step 2: Configure TRACES delivery (for spans in X-Ray)
        # ============================================================
        print(f"\n  --- TRACES (for spans in X-Ray) ---")

        # Create traces delivery source
        traces_source_name = f"{agent_type}-{short_suffix}-trace-src"
        print(f"  Source name: {traces_source_name} ({len(traces_source_name)} chars)")
        try:
            logs_client.put_delivery_source(
                name=traces_source_name, resourceArn=runtime_arn, logType="TRACES"
            )
            print(f"  ✅ Created TRACES delivery source")
            results["traces_source"] = traces_source_name
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Traces source already exists")
                results["traces_source"] = traces_source_name
            else:
                raise

        # Create traces delivery destination (X-Ray)
        traces_dest_name = f"{agent_type}-{short_suffix}-trace-dst"
        try:
            logs_client.put_delivery_destination(
                name=traces_dest_name, deliveryDestinationType="XRAY"
            )
            print(f"  ✅ Created TRACES delivery destination (XRAY)")
            results["traces_destination"] = traces_dest_name
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Traces destination already exists")
                results["traces_destination"] = traces_dest_name
            else:
                raise

        # Create traces delivery connection
        try:
            dest_info = logs_client.get_delivery_destination(name=traces_dest_name)
            dest_arn = dest_info["deliveryDestination"]["arn"]
            logs_client.create_delivery(
                deliverySourceName=traces_source_name, deliveryDestinationArn=dest_arn
            )
            print(f"  ✅ Created TRACES delivery connection")
            results["traces_delivery"] = "created"
        except ClientError as e:
            if "ConflictException" in str(e) or "already exists" in str(e).lower():
                print(f"  ℹ️  Traces delivery connection already exists")
                results["traces_delivery"] = "existing"
            else:
                raise

        return results

    except Exception as e:
        print(f"  ❌ Error: {e}")
        return results


# Configure log delivery for all observability agents
print("=" * 60)
print("CONFIGURING CLOUDWATCH LOG DELIVERY FOR OBSERVABILITY")
print("=" * 60)
print("\nThis configures BOTH:")
print("  1. APPLICATION_LOGS → CloudWatch Log Group (for GenAI events)")
print("  2. TRACES → X-Ray (for spans)")

LOG_DELIVERY_RESULTS = {}
for agent_type, runtime_info in OBSERVABILITY_RUNTIMES.items():
    results = configure_log_delivery_for_agent(agent_type, runtime_info)
    LOG_DELIVERY_RESULTS[agent_type] = results
    time.sleep(1)

print("\n" + "=" * 60)
print("LOG DELIVERY CONFIGURATION SUMMARY")
print("=" * 60)
for agent_type, results in LOG_DELIVERY_RESULTS.items():
    has_logs = (
        results.get("logs_source")
        and results.get("logs_destination")
        and results.get("logs_delivery")
    )
    has_traces = (
        results.get("traces_source")
        and results.get("traces_destination")
        and results.get("traces_delivery")
    )
    logs_status = "✅" if has_logs else "❌"
    traces_status = "✅" if has_traces else "❌"
    print(f"  {agent_type.upper()}:")
    print(f"    {logs_status} APPLICATION_LOGS (GenAI events)")
    print(f"    {traces_status} TRACES (spans)")

CONFIGURING CLOUDWATCH LOG DELIVERY FOR OBSERVABILITY

This configures BOTH:
  1. APPLICATION_LOGS → CloudWatch Log Group (for GenAI events)
  2. TRACES → X-Ray (for spans)

CONFIGURING LOG DELIVERY: ORDER AGENT
  Runtime ID: ecommerce_workshop_order_agent_observability-6Av3mF4GAl
  Short suffix: 6Av3mF4GAl
  ✅ Created vended log group: /aws/vendedlogs/bedrock-agentcore/ecommerce_workshop_order_agent_observability-6Av3mF4GAl

  --- APPLICATION_LOGS (for GenAI events) ---
  Source name: order-6Av3mF4GAl-logs-src (25 chars)
  ✅ Created APPLICATION_LOGS delivery source
  ✅ Created APPLICATION_LOGS delivery destination (CWL)
  ✅ Created APPLICATION_LOGS delivery connection

  --- TRACES (for spans in X-Ray) ---
  Source name: order-6Av3mF4GAl-trace-src (26 chars)
  ✅ Created TRACES delivery source
  ✅ Created TRACES delivery destination (XRAY)
  ❌ Error: An error occurred (ValidationException) when calling the CreateDelivery operation: X-Ray Delivery Destination is supported with CloudWatc

## Step 8: Test Multi-Agent Orchestration and Generate Traces

Now let's invoke the orchestrator to test the full multi-agent system and generate trace data for CloudWatch.


In [15]:
def invoke_observability_orchestrator(query: str) -> dict:
    """
    Invoke the observability-enabled orchestrator agent and capture session ID for trace lookup.
    """
    # Generate unique session ID for trace lookup
    session_id = f"observability-test-{int(time.time())}-{uuid.uuid4().hex[:16]}"

    print(f"\n{'=' * 60}")
    print(f"INVOKING OBSERVABILITY ORCHESTRATOR")
    print(f"{'=' * 60}")
    print(f"Query: {query}")
    print(f"Session ID: {session_id}")

    start_time = time.time()

    try:
        payload = json.dumps({"prompt": query})

        # Use the NEW observability orchestrator
        orchestrator_arn = OBSERVABILITY_RUNTIMES["orchestrator"]["arn"]

        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=orchestrator_arn,
            runtimeSessionId=session_id,
            payload=payload.encode("utf-8"),
            qualifier="DEFAULT",
        )

        elapsed_time = time.time() - start_time

        response_body = response.get("response", b"")
        if hasattr(response_body, "read"):
            response_body = response_body.read()

        response_data = json.loads(response_body)

        print(f"\n✅ Response received in {elapsed_time:.2f}s")

        # Extract response message
        if isinstance(response_data, dict):
            output = response_data.get("output", response_data)
            if isinstance(output, dict):
                message = output.get("message", str(output))
                if "agent" in output:
                    print(f"Agent: {output['agent']}")
                if "routed_to" in output:
                    print(f"Routed to: {output.get('routed_to', [])}")
                if "tools_used" in output:
                    print(f"Tools used: {output.get('tools_used', 0)}")
            else:
                message = str(output)
            print(
                f"\nResponse: {message[:400]}..."
                if len(str(message)) > 400
                else f"\nResponse: {message}"
            )

        return {
            "success": True,
            "session_id": session_id,
            "elapsed_time": elapsed_time,
            "response": response_data,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        return {
            "success": False,
            "session_id": session_id,
            "error": str(e),
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }


# Test queries that exercise the full multi-agent system with MCP tools
TEST_QUERIES = [
    "What is the status of order ORD-2024-10001?",  # Order Agent + Tools
    "Do you have any wireless headphones under $100?",  # Product Agent + Tools
    "What are the benefits of Gold membership?",  # Account Agent + Tools
    "Show me order ORD-2024-10002 status and recommend wireless products under $150",  # Multi-domain
]

TEST_RESULTS = []

In [16]:
# Run test queries to generate traces
print("Testing observability-enabled multi-agent orchestration...")
print("Note: Traces may take 1-2 minutes to appear in CloudWatch\n")

if "orchestrator" in OBSERVABILITY_RUNTIMES and agents_ready:
    for i, query in enumerate(TEST_QUERIES, 1):
        print(f"\n{'#' * 60}")
        print(f"TEST {i}/{len(TEST_QUERIES)}")
        print(f"{'#' * 60}")

        result = invoke_observability_orchestrator(query)
        TEST_RESULTS.append(result)
        time.sleep(3)
else:
    print("❌ Orchestrator not available. Cannot run tests.")
    if "orchestrator" not in OBSERVABILITY_RUNTIMES:
        print("   Orchestrator runtime was not created.")
    if not agents_ready:
        print("   Agents are not ready yet.")

# Summary
print("\n" + "=" * 60)
print("TEST RESULTS SUMMARY")
print("=" * 60)
successful = sum(1 for r in TEST_RESULTS if r.get("success"))
print(f"\n{successful}/{len(TEST_RESULTS)} tests successful\n")
for i, result in enumerate(TEST_RESULTS, 1):
    status = "✅" if result.get("success") else "❌"
    elapsed = result.get("elapsed_time", 0)
    print(f"  {status} Test {i}: {elapsed:.2f}s")

Testing observability-enabled multi-agent orchestration...
Note: Traces may take 1-2 minutes to appear in CloudWatch


############################################################
TEST 1/4
############################################################

INVOKING OBSERVABILITY ORCHESTRATOR
Query: What is the status of order ORD-2024-10001?
Session ID: observability-test-1771910732-1d2ec5c462314fee

✅ Response received in 5.27s

############################################################
TEST 2/4
############################################################

INVOKING OBSERVABILITY ORCHESTRATOR
Query: Do you have any wireless headphones under $100?
Session ID: observability-test-1771910741-85dca24793804b73

✅ Response received in 6.92s

############################################################
TEST 3/4
############################################################

INVOKING OBSERVABILITY ORCHESTRATOR
Query: What are the benefits of Gold membership?
Session ID: observability-test-1771910750-

## Step 9: Validate Traces in CloudWatch

Now we programmatically verify that traces and spans are being captured in CloudWatch. We'll wait for traces to propagate and then query X-Ray and CloudWatch Logs.


In [17]:
# Wait for traces to propagate to CloudWatch
print("Waiting 60 seconds for traces to propagate to CloudWatch...")
print("(Traces typically take 1-2 minutes to appear after invocation)")
time.sleep(60)


def query_spans_log_group(time_range_minutes: int = 30) -> dict:
    """
    Query the aws/spans log group for OTEL spans.

    AgentCore traces are delivered to CloudWatch via log delivery (not traditional X-Ray).
    The spans appear in the 'aws/spans' log group and are displayed in
    CloudWatch GenAI Observability dashboard.
    """
    print("\nQuerying aws/spans log group for OTEL traces...")

    results = {
        "total_spans": 0,
        "span_types": {},
        "services": set(),
        "sample_spans": [],
    }

    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int(
        (datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp()
        * 1000
    )

    # The spans log group where TRACES delivery sends data
    spans_log_groups = ["aws/spans", "aws/spans/default"]

    for log_group in spans_log_groups:
        try:
            # Get recent log streams
            streams_response = logs_client.describe_log_streams(
                logGroupName=log_group,
                orderBy="LastEventTime",
                descending=True,
                limit=10,
            )

            streams = streams_response.get("logStreams", [])
            if not streams:
                continue

            print(f"  Found {len(streams)} log streams in {log_group}")

            for stream in streams:
                stream_name = stream["logStreamName"]

                # Get events from this stream
                events_response = logs_client.get_log_events(
                    logGroupName=log_group,
                    logStreamName=stream_name,
                    startTime=start_time,
                    endTime=end_time,
                    limit=200,
                )

                events = events_response.get("events", [])

                for event in events:
                    message = event.get("message", "")
                    try:
                        span_data = json.loads(message)
                        results["total_spans"] += 1

                        # Extract span name/type
                        span_name = span_data.get("name", "unknown")
                        results["span_types"][span_name] = (
                            results["span_types"].get(span_name, 0) + 1
                        )

                        # Extract service name from scope or attributes
                        scope = span_data.get("scope", {})
                        service = scope.get("name", "")
                        if service:
                            results["services"].add(service)

                        # Store sample spans
                        if len(results["sample_spans"]) < 5:
                            results["sample_spans"].append(
                                {
                                    "name": span_name,
                                    "traceId": span_data.get("traceId", "")[:20],
                                    "service": service,
                                }
                            )
                    except json.JSONDecodeError:
                        pass

            if results["total_spans"] > 0:
                print(f"  ✅ Found {results['total_spans']} spans in {log_group}")
                break

        except logs_client.exceptions.ResourceNotFoundException:
            print(f"  Log group {log_group} not found")
        except Exception as e:
            print(f"  Error querying {log_group}: {str(e)[:50]}")

    return results


def query_vendedlogs_for_genai_events(time_range_minutes: int = 30) -> dict:
    """
    Query vendedlogs for GenAI events (APPLICATION_LOGS delivery).
    These contain detailed GenAI semantic convention events.
    """
    print("\nQuerying vendedlogs for GenAI events...")

    results = {"total_events": 0, "agents_with_events": []}

    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int(
        (datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp()
        * 1000
    )

    for agent_type, runtime_info in OBSERVABILITY_RUNTIMES.items():
        runtime_id = runtime_info["id"]
        log_group = f"/aws/vendedlogs/bedrock-agentcore/{runtime_id}"

        try:
            streams_response = logs_client.describe_log_streams(
                logGroupName=log_group,
                orderBy="LastEventTime",
                descending=True,
                limit=5,
            )

            agent_events = 0
            for stream in streams_response.get("logStreams", []):
                stream_name = stream["logStreamName"]
                events_response = logs_client.get_log_events(
                    logGroupName=log_group,
                    logStreamName=stream_name,
                    startTime=start_time,
                    endTime=end_time,
                    limit=100,
                )
                agent_events += len(events_response.get("events", []))

            if agent_events > 0:
                results["total_events"] += agent_events
                results["agents_with_events"].append(f"{agent_type}: {agent_events}")
                print(f"  ✅ {agent_type}: {agent_events} events")

        except logs_client.exceptions.ResourceNotFoundException:
            pass
        except Exception as e:
            pass

    return results


def query_runtime_logs(time_range_minutes: int = 30) -> dict:
    """Query agent runtime logs for application logs."""
    print("\nQuerying runtime logs...")

    results = {"total_events": 0, "agents_with_logs": []}

    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int(
        (datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp()
        * 1000
    )

    for agent_type, runtime_info in OBSERVABILITY_RUNTIMES.items():
        runtime_id = runtime_info["id"]
        log_group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"

        try:
            streams_response = logs_client.describe_log_streams(
                logGroupName=log_group,
                orderBy="LastEventTime",
                descending=True,
                limit=3,
            )

            agent_events = 0
            for stream in streams_response.get("logStreams", []):
                stream_name = stream["logStreamName"]
                events_response = logs_client.get_log_events(
                    logGroupName=log_group,
                    logStreamName=stream_name,
                    startTime=start_time,
                    endTime=end_time,
                    limit=50,
                )
                agent_events += len(events_response.get("events", []))

            if agent_events > 0:
                results["total_events"] += agent_events
                results["agents_with_logs"].append(f"{agent_type}: {agent_events}")
                print(f"  ✅ {agent_type}: {agent_events} log entries")

        except logs_client.exceptions.ResourceNotFoundException:
            pass
        except Exception as e:
            pass

    return results


# Run validation
print("=" * 60)
print("VALIDATING OBSERVABILITY DATA IN CLOUDWATCH")
print("=" * 60)
print("\nNote: AgentCore traces are delivered to CloudWatch GenAI Observability")
print("      via log delivery, not traditional X-Ray trace storage.")

# Query all data sources
SPANS_RESULTS = query_spans_log_group(time_range_minutes=30)
VENDEDLOGS_RESULTS = query_vendedlogs_for_genai_events(time_range_minutes=30)
RUNTIME_LOGS_RESULTS = query_runtime_logs(time_range_minutes=30)

# Summary
print("\n" + "=" * 60)
print("OBSERVABILITY VALIDATION SUMMARY")
print("=" * 60)

spans_count = SPANS_RESULTS.get("total_spans", 0)
genai_events = VENDEDLOGS_RESULTS.get("total_events", 0)
runtime_logs = RUNTIME_LOGS_RESULTS.get("total_events", 0)

print(f"\n🔗 OTEL Spans (aws/spans): {spans_count} spans")
if SPANS_RESULTS.get("span_types"):
    print("   Span types found:")
    for span_type, count in list(SPANS_RESULTS["span_types"].items())[:5]:
        print(f"     • {span_type}: {count}")

print(f"\n📝 GenAI Events (vendedlogs): {genai_events} events")
if VENDEDLOGS_RESULTS.get("agents_with_events"):
    for agent_info in VENDEDLOGS_RESULTS["agents_with_events"]:
        print(f"     • {agent_info}")

print(f"\n📋 Runtime Logs: {runtime_logs} entries")

total_data = spans_count + genai_events + runtime_logs
print(f"\n📈 Total Observability Data: {total_data} records")

print(f"\n🔗 View in CloudWatch GenAI Observability Dashboard:")
print(
    f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore"
)

if total_data > 0:
    print("\n✅ Observability is working! Data is being captured in CloudWatch.")
    print("\n📋 What you'll see in the GenAI Observability dashboard:")
    print("   • invoke_agent spans - Agent invocation with input/output")
    print("   • chat spans - LLM model calls with token usage")
    print("   • execute_tool spans - Tool calls with inputs/outputs")
    print("   • execute_event_loop_cycle spans - Agent reasoning steps")
else:
    print("\n⚠️ No observability data found yet. Possible reasons:")
    print("   • Data may still be propagating (wait 2-3 more minutes)")
    print("   • Log delivery configuration may need time to activate")
    print("   • Verify in the CloudWatch console directly")

Waiting 60 seconds for traces to propagate to CloudWatch...
(Traces typically take 1-2 minutes to appear after invocation)
VALIDATING OBSERVABILITY DATA IN CLOUDWATCH

Note: AgentCore traces are delivered to CloudWatch GenAI Observability
      via log delivery, not traditional X-Ray trace storage.

Querying aws/spans log group for OTEL traces...
  Log group aws/spans not found
  Log group aws/spans/default not found

Querying vendedlogs for GenAI events...
  ✅ orchestrator: 4 events

Querying runtime logs...
  ✅ orchestrator: 50 log entries

OBSERVABILITY VALIDATION SUMMARY

🔗 OTEL Spans (aws/spans): 0 spans

📝 GenAI Events (vendedlogs): 4 events
     • orchestrator: 4

📋 Runtime Logs: 50 entries

📈 Total Observability Data: 54 records

🔗 View in CloudWatch GenAI Observability Dashboard:
   https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#genai-observability:bedrockAgentCore

✅ Observability is working! Data is being captured in CloudWatch.

📋 What you'll see 